# 4. Model CNN-SVM

Model má dve časti, ktoré sa trénujú oddelene:
1. **CNN** (ResNet18 z notebooku 2) prevedie každú snímku na 512 príznakov,
2. **SVM** (support vector machine) sa učí oddeliť triedy v priestore príznakov.

SVM potrebuje jeden vektor na klip, preto príznaky 16 snímok **spriemerujeme v čase**.
Poradie snímok sa tým stráca – model vidí, *čo* je v klipe, ale nie *ako sa to hýbe*.

**Protokol hodnotenia** (rovnaký pre všetky modely):
- 10 delení train/test podľa zdrojového videa,
- v každom delení sa kernel (lineárny / RBF) a hyperparametre vyberú 5-násobnou krížovou
  validáciou **iba na trénovacích dátach**, test sa použije len na finálne hodnotenie,
- hlavná metrika je **balanced accuracy** (priemer recall oboch tried), lebo triedy sú
  nevyvážené (108 eat, 164 drink); model, ktorý vždy povie „drink“, by mal accuracy 0,6.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

from har_utils import CLASSES, evaluate_svm, summarize

df = pd.read_csv("index.csv")
feats = np.load("features_resnet.npy")     # (klipy, 16, 512)
X = feats.mean(axis=1)                     # priemer v čase -> (klipy, 512)
print(X.shape)

In [ ]:
results, y_all, p_all = evaluate_svm(X, df)

Path("results").mkdir(exist_ok=True)
results.to_csv("results/cnn_svm.csv", index=False)
print(results.round(3).to_string())
print()
print(summarize(results, "CNN-SVM").to_string())

## Chyby modelu

Confusion matrix je súčtom testovacích predikcií zo všetkých 10 delení.

In [ ]:
print(classification_report(y_all, p_all, target_names=CLASSES))
ConfusionMatrixDisplay.from_predictions(y_all, p_all, display_labels=CLASSES, cmap="Blues");